# Agent World Research Analysis

This notebook demonstrates how to use the Agent World SDK for academic research analysis.

## Features
- Descriptive statistics for agent populations
- Correlation analysis between behaviors and outcomes
- Emergence pattern detection (4 algorithms)
- Statistical significance testing
- Social network analysis
- Cultural evolution tracking
- Export to GraphML/GEXF/LaTeX formats

In [ ]:
# Install dependencies (uncomment if needed)
# %pip install agent-world-sdk pandas matplotlib

from agent_world_sdk import (
    AgentWorldClient,
    AnalyzeModule,
    EconomicModule,
    SocialModule,
    BehaviorModule,
    to_graphml,
    to_gexf,
    to_latex_table,
    to_latex_summary,
)
from agent_world_sdk.dataframe import to_dataframe, agents_dataframe

import json

## 1. Connect to the World Engine

In [ ]:
# Connect to a running world-engine instance
client = AgentWorldClient("http://localhost:3000")

# Fetch current world state
state = client.world.state()
print(f"World tick: {state.get('tick', 'N/A')}")
print(f"Agent count: {state.get('agent_count', 'N/A')}")

## 2. Descriptive Statistics

In [ ]:
analyze = AnalyzeModule()

# Get agent list
agents = client.agents.list()

# Compute descriptive statistics for wealth
money_values = [a.get('money', 0) for a in agents if a.get('alive', True)]
wealth_stats = analyze.descriptive_stats(money_values)
print("Wealth Statistics:")
for k, v in wealth_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Frequency distribution of agent phases
phase_dist = analyze.frequency_distribution([a.get('phase', 'unknown') for a in agents])
print("Phase Distribution:")
print(json.dumps(phase_dist, indent=2))

# Group statistics by phase
group_stats = analyze.group_statistics(agents, 'phase', 'money')
print("\nWealth by Phase:")
for phase, stats in group_stats.items():
    print(f"  {phase}: mean={stats['mean']:.2f}, std={stats['std_dev']:.2f}")

## 3. Pandas DataFrame Export

In [ ]:
# One-liner DataFrame export
df = agents_dataframe(agents)
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)

In [ ]:
# Quick describe
df[['tokens', 'money']].describe()

## 4. Correlation Analysis

In [ ]:
# Pearson correlation between behavior metrics
activity = [a.get('tasks_completed', 0) for a in agents]
wealth = [a.get('money', 0) for a in agents]

r = analyze.pearson_correlation(activity, wealth)
print(f"Pearson r (activity vs wealth): {r}")

rho = analyze.spearman_correlation(activity, wealth)
print(f"Spearman rho (activity vs wealth): {rho}")

In [ ]:
# Full correlation matrix
corr_matrix = analyze.correlation_matrix(
    agents,
    ['money', 'tokens', 'tasks_completed'],
    method='pearson'
)
print("Correlation Matrix:")
for field, correlations in corr_matrix.items():
    print(f"  {field}:")
    for other, val in correlations.items():
        print(f"    vs {other}: {val:.4f}")

In [ ]:
# Behavior-outcome correlation with interpretation
result = analyze.behavior_outcome_correlation(
    agents,
    behavior_field='tasks_completed',
    outcome_field='money'
)
print(f"Result: {result['interpretation']}")
print(f"  Pearson: {result['pearson']:.4f}")
print(f"  Spearman: {result['spearman']:.4f}")
print(f"  N: {result['sample_size']}")

## 5. Emergence Pattern Detection

In [ ]:
# Fetch historical data
history = client.world.history()

# Algorithm 1: Phase transition detection
transitions = analyze.detect_phase_transitions(
    history, 'total_money', window=10, threshold=2.0
)
print(f"Phase transitions detected: {len(transitions)}")
for t in transitions[:5]:
    print(f"  Tick {t['tick']}: z={t['z_score']:.2f} ({t['change_direction']})")

In [ ]:
# Algorithm 2: K-means clustering
clusters = analyze.detect_clustering(
    agents, ['money', 'tokens'], k=3
)
print(f"Clusters found: {len(clusters['clusters'])}")
for c in clusters['clusters']:
    print(f"  Cluster {c['id']}: {c['size']} agents")
print(f"Centroids: {clusters['centroids']}")

In [ ]:
# Algorithm 3: Multi-field emergence detection
emergence = analyze.detect_emergent_patterns(
    history,
    ['total_money', 'total_tokens'],
    window=20,
    variance_threshold=3.0
)
print(f"Emergence events: {len(emergence['emergence_events'])}")
for e in emergence['emergence_events'][:5]:
    print(f"  Tick {e['tick']}: {e['num_fields_anomalous']} fields anomalous")

In [ ]:
# Algorithm 4: Power-law detection
wealth_values = [a.get('money', 0) for a in agents if a.get('money', 0) > 0]
power_law = analyze.detect_power_law(wealth_values)
print(f"Power-law fit: {'Yes' if power_law['is_power_law'] else 'No'}")
print(f"  Exponent: {power_law['exponent']:.4f}")
print(f"  R-squared: {power_law['r_squared']:.4f}")

## 6. Statistical Significance Testing

In [ ]:
# Compare wealth between two agent phases
adult_money = [a['money'] for a in agents if a.get('phase') == 'Adult' and a.get('alive', True)]
elder_money = [a['money'] for a in agents if a.get('phase') == 'Elder' and a.get('alive', True)]

# Welch's t-test
t_result = analyze.t_test(adult_money, elder_money)
print(f"T-test: t={t_result['t_statistic']:.4f}, p={t_result['p_value']:.4f}")
print(f"  Significant at 0.05: {t_result['significant_at_005']}")

# Mann-Whitney U test (non-parametric)
u_result = analyze.mann_whitney_u(adult_money, elder_money)
print(f"\nMann-Whitney: U={u_result['u_statistic']}, p={u_result['p_value']:.4f}")
print(f"  Significant at 0.05: {u_result['significant_at_005']}")

In [ ]:
# Chi-squared test for phase distribution
from collections import Counter
phase_counts = Counter(a.get('phase', 'unknown') for a in agents)
chi2_result = analyze.chi_squared_test(list(phase_counts.values()))
print(f"Chi-squared: {chi2_result['chi2_statistic']:.4f}")
print(f"  df: {chi2_result['df']}")
print(f"  p-value: {chi2_result['p_value']:.4f}")

## 7. Social Network Analysis

In [ ]:
social = SocialModule()

# Fetch network graph
graph_data = client.export.network_graph()
nodes = graph_data.get('nodes', [])
edges = graph_data.get('edges', [])

# Network analysis
centrality = social.degree_centrality(edges)
components = social.connected_components(edges)
summary = social.community_summary(edges)

print(f"Network Summary:")
print(f"  Nodes: {summary.get('largest_component_size', 'N/A')}")
print(f"  Components: {summary['component_count']}")
print(f"  Isolated: {summary['isolated_nodes']}")

## 8. Cultural Evolution Analysis

In [ ]:
# Cultural diversity
diversity = analyze.cultural_diversity(agents)
print(f"Cultural Diversity:")
print(f"  Shannon Entropy: {diversity['shannon_entropy']:.4f}")
print(f"  Simpson Index: {diversity['simpson_index']:.4f}")
print(f"  Unique Phases: {diversity['unique_phases']}")

# Cultural evolution across time
evolution = analyze.cultural_evolution(history)
if evolution['entropy_trajectory']:
    first = evolution['entropy_trajectory'][0]
    last = evolution['entropy_trajectory'][-1]
    print(f"\nEntropy change: {first['entropy']:.4f} -> {last['entropy']:.4f}")

## 9. Agent Behavior Trajectory

In [ ]:
# Get behavior logs for a specific agent
behavior = BehaviorModule()

# Analyze survival
survival = behavior.survival_stats(agents)
print(f"Survival Rate: {survival['survival_rate']:.2%}")
print(f"Mean ticks survived: {survival['mean_ticks_survived']:.1f}")

# Strategy classification
behavior_log = client.export.behavior_log(format='json')
if behavior_log:
    strategies = behavior.strategy_classification(behavior_log)
    print(f"\nStrategy Distribution:")
    for strategy, count in strategies['strategy_distribution'].items():
        print(f"  {strategy}: {count}")

## 10. Export for Academic Use

In [ ]:
# Export network to GraphML (for Gephi/Cytoscape)
graphml_str = to_graphml(nodes, edges)
with open('network.graphml', 'w') as f:
    f.write(graphml_str)
print(f"Exported GraphML: {len(graphml_str)} bytes")

# Export to GEXF (alternative format)
gexf_str = to_gexf(nodes, edges)
with open('network.gexf', 'w') as f:
    f.write(gexf_str)
print(f"Exported GEXF: {len(gexf_str)} bytes")

In [ ]:
# Export statistics as LaTeX table
wealth_table = to_latex_table(
    [{'Phase': k, 'Mean': v['mean'], 'Std Dev': v['std_dev'], 'N': v['count']}
     for k, v in group_stats.items()],
    caption='Wealth Distribution by Agent Phase',
    label='tab:wealth_phase',
    format_str='{:.2f}'
)
print(wealth_table)

In [ ]:
# Export world summary as LaTeX
summary = analyze.world_summary(agents, history=history)
latex_summary = to_latex_summary(
    {
        'Population': summary['population'],
        'Alive': summary['alive_count'],
        'Survival Rate': f"{summary['survival_rate']:.2%}",
        'Mean Wealth': f"{summary['wealth']['mean']:.2f}",
        'Wealth Std Dev': f"{summary['wealth']['std_dev']:.2f}",
    },
    caption='World Summary Statistics',
)
print(latex_summary)

In [ ]:
# DataFrame export for further analysis
df = agents_dataframe(agents)
df.to_csv('agents_export.csv', index=False)
df.to_json('agents_export.json', orient='records', indent=2)
print(f"Exported {len(df)} agents to CSV and JSON")

## Summary

This notebook demonstrated:

| Category | Functions Used |
|---|---|
| Descriptive Stats | `descriptive_stats`, `frequency_distribution`, `group_statistics` |
| Correlation | `pearson_correlation`, `spearman_correlation`, `correlation_matrix`, `behavior_outcome_correlation` |
| Emergence Detection | `detect_phase_transitions`, `detect_clustering`, `detect_emergent_patterns`, `detect_power_law` |
| Significance Tests | `t_test`, `mann_whitney_u`, `chi_squared_test` |
| Cultural Analysis | `cultural_diversity`, `cultural_evolution` |
| Network Analysis | `degree_centrality`, `betweenness_centrality`, `connected_components`, `community_summary` |
| Behavior Analysis | `survival_stats`, `strategy_classification`, `activity_profile` |
| Export Formats | `to_graphml`, `to_gexf`, `to_latex_table`, `to_latex_summary`, `agents_dataframe` |

The SDK `analyze` module provides 25+ analysis functions for academic research.